<a href="https://colab.research.google.com/github/DanielYaari28/kaggle-philharmonia/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

RAW = '/content/drive/MyDrive/kaggle-philharmonia/data/raw'

Mounted at /content/drive


In [2]:
import pandas as pd

train   = pd.read_csv(f'{RAW}/train.csv')
test    = pd.read_csv(f'{RAW}/test.csv')
account = pd.read_csv(f'{RAW}/account.csv', encoding='latin-1',
                      dtype={'shipping.zip.code': str, 'billing.zip.code': str})
tickets = pd.read_csv(f'{RAW}/tickets_all.csv')
subs    = pd.read_csv(f'{RAW}/subscriptions.csv')
concerts      = pd.read_csv(f'{RAW}/concerts.csv', encoding='utf-8-sig')
concerts_1415 = pd.read_csv(f'{RAW}/concerts_2014-15.csv', encoding='utf-8-sig')
zipcodes      = pd.read_csv(f'{RAW}/zipcodes.csv', lineterminator='\r')
sample_sub    = pd.read_csv(f'{RAW}/sample_submission.csv')

In [3]:
# Ensure all files are corretly uploaded
expected = {
    'train': (6941, 2), 'test': (2975, 1), 'account': (19833, 10),
    'tickets': (2808, 8), 'subs': (28627, 9), 'concerts': (103, 6),
    'concerts_1415': (6, 5), 'zipcodes': (42522, 12), 'sample_sub': (2975, 2),
}
for name, shape in expected.items():
    actual = globals()[name].shape
    print(f"{name:14} {str(actual):14} {'OK' if actual == shape else '<-- MISMATCH, expected ' + str(shape)}")

print("\nconcerts columns:", concerts.columns.tolist()[:2])   # should start with 'season', no BOM junk
print("account zip dtype:", account['billing.zip.code'].dtype)  # should be object/str, not float

train          (6941, 2)      OK
test           (2975, 1)      OK
account        (19833, 10)    OK
tickets        (2808, 8)      OK
subs           (28627, 9)     OK
concerts       (103, 6)       OK
concerts_1415  (6, 5)         OK
zipcodes       (42522, 12)    OK
sample_sub     (2975, 2)      OK

concerts columns: ['season', 'concert.name']
account zip dtype: object


In [4]:
for name in ['account', 'tickets', 'subs', 'concerts', 'zipcodes']:
    df = globals()[name]
    na = df.isna().mean().round(3)
    print(f"\n== {name}\n{na[na > 0].sort_values(ascending=False)}")


== account
shipping.zip.code    0.987
shipping.city        0.986
relationship         0.967
first.donated        0.721
billing.zip.code     0.149
billing.city         0.112
dtype: float64

== tickets
marketing.source    0.792
price.level         0.074
location            0.016
set                 0.008
dtype: float64

== subs
section        0.159
price.level    0.123
dtype: float64

== concerts
location    0.029
dtype: float64

== zipcodes
TotalWages             0.322
EstimatedPopulation    0.321
TaxReturnsFiled        0.321
Lat                    0.015
Long                   0.015
dtype: float64


In [8]:
print("missing first.donated:", fd_missing.sum())
print("…of those with 0 donations:", (account.loc[fd_missing, 'no.donations.lifetime'] == 0).sum())
print("accounts with 0 donations but a date:", (~fd_missing & (account['no.donations.lifetime'] == 0)).sum())

missing first.donated: 14298
…of those with 0 donations: 14298
accounts with 0 donations but a date: 0


missing here means that they have never donated

In [9]:
print(subs.groupby('location')['section'].apply(lambda s: s.isna().mean()).round(3))
print(subs.groupby('season')['section'].apply(lambda s: s.isna().mean()).round(3))

location
Berkeley Saturday    0.166
Berkeley Sunday      0.177
Contra Costa         0.138
Los Angeles          0.000
Orange County        0.005
Peninsula            0.173
San Francisco        0.173
Santa Rosa           0.000
Name: section, dtype: float64
season
1993-1994    0.000
1994-1995    0.000
1995-1996    0.000
1996-1997    0.002
1997-1998    0.000
1998-1999    0.000
1999-2000    0.001
2000-2001    0.001
2001-2002    0.002
2002-2003    1.000
2003-2004    1.000
2004-2005    1.000
2005-2006    0.006
2006-2007    0.006
2007-2008    0.000
2008-2009    0.000
2009-2010    0.000
2010-2011    0.023
2011-2012    0.000
2012-2013    0.004
2013-2014    1.000
Name: section, dtype: float64


So this looks like a record-keeping gap. Section wasn't recorded in those seasons, and the blanks don't carry information about the subscriber. Missing here means unknown...

In [10]:
print(subs.groupby('season')['price.level'].apply(lambda s: s.isna().mean()).round(3))

season
1993-1994    0.000
1994-1995    0.000
1995-1996    0.000
1996-1997    0.002
1997-1998    0.000
1998-1999    0.000
1999-2000    0.001
2000-2001    0.001
2001-2002    0.002
2002-2003    1.000
2003-2004    1.000
2004-2005    1.000
2005-2006    0.006
2006-2007    0.006
2007-2008    0.000
2008-2009    0.000
2009-2010    0.000
2010-2011    0.000
2011-2012    0.000
2012-2013    0.001
2013-2014    0.000
Name: price.level, dtype: float64


Section and price level were both unrecorded in 2002–05, and section was also unrecorded in 2013–14. Missing values in these columns mean "unknown," not "none."

In [11]:
# 1. Does each venue have sections in the seasons where they're recorded?
print(subs.dropna(subset=['section']).groupby('location')['section'].unique())

# 2. Do price levels still vary in 2013-14?
print(subs[subs.season == '2013-2014']['price.level'].value_counts())

# 3. Where does 'GA' show up in tickets?
print(tickets[tickets['price.level'] == 'GA'][['season', 'location', 'set']])

location
Berkeley Saturday    [Balcony Front, Balcony Rear, Orchestra, Premi...
Berkeley Sunday      [Balcony Rear, Balcony Front, Orchestra, Orche...
Contra Costa         [Orchestra, Balcony, Premium Orchestra, Galler...
Los Angeles                                                [Orchestra]
Orange County        [Orchestra, Balcony, Orchestra Rear, Orchestra...
Peninsula            [Orchestra, Balcony, Premium Orchestra, Orches...
San Francisco        [Premium Orchestra, Orchestra, Orchestra Front...
Santa Rosa                            [Orchestra, Balcony, Santa Rosa]
Name: section, dtype: object
price.level
1.0    301
0.0    274
2.0    149
4.0    134
3.0    124
Name: count, dtype: int64
         season        location  set
357   2011-2012  Family concert  3.0
393   2011-2012  Family concert  3.0
688   2011-2012  Family concert  3.0
1823  2011-2012  Family concert  3.0
2637  2011-2012  Family concert  3.0
2765  2011-2012  Family concert  3.0


Subscription section and price level are missing because of record-keeping gaps in 2002–05 (both) and 2013–14 (section only). General admission applies only to the Family concert tickets.

In [12]:
pd.crosstab(subs['season'], subs['price.level'], normalize='index').round(2).tail(8)

price.level,0.0,1.0,2.0,3.0,4.0
season,,,,,
2006-2007,0.00,0.60,0.14,0.18,0.09
2007-2008,0.00,0.59,0.14,0.19,0.08
2008-2009,0.00,0.61,0.13,0.19,0.07
2009-2010,0.00,0.60,0.12,0.19,0.09
2010-2011,0.28,0.31,0.16,0.10,0.14
2011-2012,0.26,0.33,0.15,0.12,0.14
2012-2013,0.25,0.32,0.18,0.11,0.14
2013-2014,0.28,0.31,0.15,0.13,0.14


In [13]:
recent = subs[subs.season.isin(['2010-2011', '2011-2012', '2012-2013'])]
pd.crosstab(recent['section'], recent['price.level'])

price.level,0.0,1.0,2.0,3.0,4.0
section,,,,,
Balcony,0,0,1,0,0
Balcony Front,0,0,265,145,67
Balcony Rear,0,0,0,0,142
Box,0,13,0,0,0
Boxes House Left,0,11,0,0,0
Boxes House Right,0,13,0,0,0
Dress Circle,0,0,82,17,11
Floor,0,1,0,0,0
Orchestra,824,958,164,189,215


In [14]:
pd.set_option('display.max_rows', 100)
pd.crosstab(recent['section'], recent['price.level'])

price.level,0.0,1.0,2.0,3.0,4.0
section,,,,,
Balcony,0,0,1,0,0
Balcony Front,0,0,265,145,67
Balcony Rear,0,0,0,0,142
Box,0,13,0,0,0
Boxes House Left,0,11,0,0,0
Boxes House Right,0,13,0,0,0
Dress Circle,0,0,82,17,11
Floor,0,1,0,0,0
Orchestra,824,958,164,189,215


In [5]:
display(account.head())
display(account.describe())
account['relationship'].value_counts(dropna=False)

,account.id,shipping.zip.code,billing.zip.code,shipping.city,billing.city,relationship,amount.donated.2013,amount.donated.lifetime,no.donations.lifetime,first.donated
0,001i000000LhSrQ,94102,94102,San Francisco,San Francisco,NaN,0.0,0.0,0,NaN
1,001i000000LhyPD,NaN,94303,NaN,Palo Alto,NaN,0.0,500.0,5,1987-07-04 00:00:00
2,001i000000LhyPE,NaN,94597,NaN,Walnut Creek,NaN,0.0,783.0,7,2008-06-02 00:00:00
3,001i000000LhyPF,NaN,94610,NaN,Oakland,NaN,1500.0,28435.0,28,1986-07-04 00:00:00
4,001i000000LhyPG,NaN,94024,NaN,Los Altos,NaN,300.0,4969.0,16,1997-04-07 00:00:00


,amount.donated.2013,amount.donated.lifetime,no.donations.lifetime
count,19833.000000,1.983300e+04,19833.000000
mean,85.204938,1.605946e+03,1.631473
std,2224.581861,3.312114e+04,4.974458
min,0.000000,0.000000e+00,0.000000
25%,0.000000,0.000000e+00,0.000000
50%,0.000000,0.000000e+00,0.000000
75%,0.000000,2.500000e+01,1.000000
max,211400.000000,2.287650e+06,95.000000


,count
relationship,
NaN,19172
Education Participant,422
Vendor - Artistic,189
Vendor - General,13
Vendor - Marketing,12
Peer Organization,10
Other,9
Corporation,3
Vendor - Artistic; Peer Organization,1


In [6]:
z = account['billing.zip.code'].dropna()
print(z.str.len().value_counts())
z[~z.str.fullmatch(r'\d{5}(-\d{4})?')].head(20)

billing.zip.code
5     16202
10      479
4       164
7        19
6         8
8         3
3         2
9         1
Name: count, dtype: int64


,billing.zip.code
811,944041913
1117,3755
1140,T8N 3Z9
1141,T2L 2L2
1357,4469
1358,2116
1390,6106
1408,7102
1415,AL5 9FA
1417,A-5010
